## Init

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import from_json, col, explode, arrays_zip, to_timestamp, to_date, hour, current_timestamp, max as spark_max
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, DoubleType

## Define structures for the 'hourly' JSON and the complete payload 

In [0]:
hourly_schema = StructType([
    StructField("time", ArrayType(StringType())),
    StructField("temperature_2m", ArrayType(DoubleType())),
    StructField("relative_humidity_2m", ArrayType(DoubleType())),
    StructField("precipitation", ArrayType(DoubleType()))
])

payload_schema = StructType([
    StructField("hourly", hourly_schema)
])

## Read from bronze table

In [0]:
# Get the timestamp of the most recent ingestion in bronze
latest_ingestion = (
    spark.read
        .table("weather.bronze.data_raw")
        .select(spark_max("ingestion_timestamp"))
        .collect()[0][0]
)

# Read only the last ingestion
df_bronze_read = (
    spark.read
        .table("weather.bronze.data_raw")
        .filter(col("ingestion_timestamp") == latest_ingestion)
)

## Un-nest the raw payload

In [0]:
df_parsed = (
    df_bronze_read
    .withColumn(
        "parsed_json", 
        from_json(col("raw_payload"), payload_schema)
    )
)

## Create a new row for each weather measurement 

In [0]:
df_zipped = (
    df_parsed
    .withColumn(
        "zipped_data", 
        explode(arrays_zip(
            col("parsed_json.hourly.time"),
            col("parsed_json.hourly.temperature_2m"),
            col("parsed_json.hourly.relative_humidity_2m"),
            col("parsed_json.hourly.precipitation")
        ))
    )
)

## Get date and hour of each record

In [0]:
df_date_hour = (
    df_zipped
    .withColumn(
        "date",
        to_date("zipped_data.time") 
    )
    .withColumn(
        "hour",
        hour("zipped_data.time") 
    )
)

## Select final columns

In [0]:
df_silver_clean = (
    df_date_hour
    .select(
        col("city_name"),
        col("state_name"),
        col("date"),
        col("hour"),
        to_timestamp(col("zipped_data.time")).alias("weather_timestamp"),
        col("zipped_data.temperature_2m").alias("temperature_celsius"),
        col("zipped_data.relative_humidity_2m").alias("humidity_percentage"),
        col("zipped_data.precipitation").alias("precipitation_mm"),
        current_timestamp().alias("processed_at")
    ).dropDuplicates(["city_name", "weather_timestamp"])
)


## Write in silver table with merge (UPSERT)

In [0]:
# A. Initialize the silver table in Delta Lake if it is the first time it is being executed 
(
    df_silver_clean.write
        .mode("ignore")
        .format("delta")
        .saveAsTable("weather.silver.weather_hourly")
)

In [0]:
# B. Execute MERGE using (city_name + weather_timestamp) as unique key 
target_table = DeltaTable.forName(spark, "weather.silver.weather_hourly")

target_table.alias("target").merge(
    df_silver_clean.alias("source"),
    """
    target.city_name = source.city_name AND 
    target.weather_timestamp = source.weather_timestamp
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()